# SEQNN: Hybrid Quantum Deep Learning for Earth Observation

**Paper:** Fan et al., "Hybrid Quantum Deep Learning With Superpixel Encoding for Earth Observation Data Classification", IEEE TNNLS, Vol. 36, No. 6, June 2025

**This notebook includes:**
- Corrected implementation matching the paper
- GPU optimization support
- Proper quantum circuit structure
- Complete training pipeline

## 1. Setup & Installation

In [ ]:
# Install required packages
# Uncomment the lines below if packages are not installed

# !pip install torch torchvision
# !pip install pennylane
# !pip install scikit-learn scipy matplotlib

# For GPU support (optional):
# !pip install pennylane-lightning[gpu]  # Requires CUDA
# !pip install jax[cuda12_pip]           # For JAX GPU backend

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Import SEQNN modules (use corrected versions)
from seqnn_pytorch_corrected import (
    SEQNN, 
    SEQNNTrainer, 
    SEQNNConfig,
    build_SEQNN_model,
    set_seed
)
from seqnn_dataLoader_corrected import DataLoader

# Set random seed for reproducibility
set_seed(42)

## 2. Load Dataset

Supported datasets from the paper:
- `'sat'`: SAT-6 (4200 train, 1200 valid, 1200 test, 6 classes)
- `'lcz'`: So2Sat LCZ42 (5 semantic classes)
- `'overhead'`: Overhead-MNIST (5 classes)
- `'synthetic'`: Synthetic data for testing

In [ ]:
# Configuration
DATASET = 'synthetic'  # Change to 'sat', 'lcz', or 'overhead' if you have the data

# Load data
print(f"Loading {DATASET} dataset...")
dataloader = DataLoader(DATASET)

# Display dataset info
info = dataloader.get_info()
print(f"\nDataset Information:")
for key, value in info.items():
    print(f"  {key}: {value}")

In [ ]:
# Get data with one-hot encoded labels
train_x, train_y, valid_x, valid_y, test_x, test_y = dataloader.get_data()

print(f"\nData shapes:")
print(f"  Training:   X={train_x.shape}, Y={train_y.shape}")
print(f"  Validation: X={valid_x.shape}, Y={valid_y.shape}")
print(f"  Test:       X={test_x.shape}, Y={test_y.shape}")

# Get dimensions for model
n_channels = train_x.shape[-1]
n_classes = train_y.shape[-1]
print(f"\n  Channels: {n_channels}, Classes: {n_classes}")

In [ ]:
# Visualize sample images
def visualize_samples(images, labels, categories, n_samples=5):
    """Display sample images with labels."""
    fig, axes = plt.subplots(1, n_samples, figsize=(15, 3))
    
    for i in range(n_samples):
        img = images[i]
        label_idx = np.argmax(labels[i])
        label_name = categories[label_idx] if label_idx < len(categories) else str(label_idx)
        
        # Display image (use first 3 channels for RGB)
        if img.shape[-1] >= 3:
            display_img = img[:, :, :3]
        else:
            display_img = img[:, :, 0]
        
        axes[i].imshow(display_img, cmap='gray' if display_img.ndim == 2 else None)
        axes[i].set_title(f"Class: {label_name}")
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

categories = dataloader.get_categories()
visualize_samples(train_x, train_y, categories)

## 3. Build SEQNN Model

Model architecture (from paper Figure 1):
1. **Superpixel Preprocessing**: Patches → FC → ReLU
2. **Quantum Encoding**: Controlled U3 gates + CZ entanglement
3. **Quantum Convolution**: 144 trainable parameters
4. **Measurement**: X-basis (64 features)
5. **Classifier**: Dense + Softmax

In [ ]:
# Configuration
USE_QUANTUM = False  # Set True for actual quantum simulation (slower)
USE_GPU = torch.cuda.is_available()

# Build model
model, trainer = build_SEQNN_model(
    n_classes=n_classes,
    n_channels=n_channels,
    dataset=DATASET,
    use_quantum=USE_QUANTUM,
    use_gpu=USE_GPU
)

In [ ]:
# Detailed parameter breakdown
breakdown = model.get_parameter_breakdown()

print("\nParameter Breakdown:")
print(f"  Superpixel Preprocessing: {breakdown['superpixel']:,}")
print(f"  Quantum Layer:            {breakdown['quantum']:,}")
print(f"  Classifier:               {breakdown['classifier']:,}")
print(f"  " + "-" * 35)
print(f"  Total:                    {breakdown['total']:,}")

# Paper Table VI comparison
print(f"\n  Paper Table VI (SAT-6): 1119 parameters")
print(f"  Current model:          {breakdown['total']} parameters")

## 4. Training

Training settings from paper (Section V):
- Learning rate: 0.01
- Batch size: 50
- Epochs: 200
- Optimizer: Adam

In [ ]:
# Training configuration
EPOCHS = 200      # Paper: 200 epochs
BATCH_SIZE = 50   # Paper: batch size 50
LEARNING_RATE = 0.01  # Paper: learning rate 0.01

# For quick testing, reduce epochs
QUICK_TEST = True
if QUICK_TEST:
    EPOCHS = 10
    print("Quick test mode: 10 epochs")

# Compile trainer
trainer.compile(learning_rate=LEARNING_RATE, use_scheduler=True)

print(f"\nTraining Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Device: {trainer.device}")

In [ ]:
# Train the model
print(f"\n{'='*60}")
print(f"Starting training at {datetime.now().strftime('%H:%M:%S')}")
print(f"{'='*60}\n")

history = trainer.fit(
    train_x, train_y,
    val_x=valid_x, val_y=valid_y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
    save_best=f'best_{DATASET}_model.pt',
    use_amp=USE_GPU  # Mixed precision on GPU
)

print(f"\n{'='*60}")
print(f"Training completed at {datetime.now().strftime('%H:%M:%S')}")
print(f"{'='*60}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Train Accuracy', linewidth=2)
axes[1].plot(history['val_acc'], label='Validation Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print best results
best_val_acc = max(history['val_acc'])
best_epoch = history['val_acc'].index(best_val_acc) + 1
print(f"\nBest Validation Accuracy: {best_val_acc:.4f} (Epoch {best_epoch})")

## 5. Evaluation

In [ ]:
# Evaluate on all splits
print("\n" + "="*60)
print("Final Evaluation Results")
print("="*60)

train_loss, train_acc = trainer.evaluate(train_x, train_y)
print(f"\nTraining Set:")
print(f"  Loss: {train_loss:.4f}")
print(f"  Accuracy: {train_acc:.4f}")

valid_loss, valid_acc = trainer.evaluate(valid_x, valid_y)
print(f"\nValidation Set:")
print(f"  Loss: {valid_loss:.4f}")
print(f"  Accuracy: {valid_acc:.4f}")

test_loss, test_acc = trainer.evaluate(test_x, test_y)
print(f"\nTest Set:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_acc:.4f}")

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Get predictions
predictions = trainer.predict(test_x)
y_pred = np.argmax(predictions, axis=1)
y_true = np.argmax(test_y, axis=1)

# Plot confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=categories, yticklabels=categories)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=categories))

## 6. Comparison with Paper Results

Paper Table VI - Test Accuracy:
- Overhead-MNIST: 0.913 ± 0.004
- So2Sat LCZ42: 0.914 ± 0.004
- SAT-6: 0.952 ± 0.004

In [ ]:
# Paper results (Table VI)
paper_results = {
    'overhead': {'accuracy': 0.913, 'std': 0.004, 'params': 622},
    'lcz': {'accuracy': 0.914, 'std': 0.004, 'params': 1054},
    'sat': {'accuracy': 0.952, 'std': 0.004, 'params': 1119},
}

print("\nComparison with Paper Results (Table VI):")
print("="*60)

if DATASET in paper_results:
    paper = paper_results[DATASET]
    print(f"\nDataset: {DATASET.upper()}")
    print(f"  Paper Accuracy:  {paper['accuracy']:.3f} ± {paper['std']:.3f}")
    print(f"  Our Accuracy:    {test_acc:.3f}")
    print(f"  Paper Parameters: {paper['params']}")
    print(f"  Our Parameters:   {model.count_parameters()}")
    
    if test_acc >= paper['accuracy'] - 2*paper['std']:
        print(f"\n  ✓ Results within expected range!")
    else:
        print(f"\n  Note: Results may improve with full training (200 epochs)")
else:
    print(f"\nDataset '{DATASET}' not in paper comparison table.")
    print(f"Our Test Accuracy: {test_acc:.4f}")

## 7. Save Model

In [ ]:
# Save the final model
import os

save_dir = 'trained_models'
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, f'seqnn_{DATASET}_final.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'config': model.config.__dict__,
    'n_classes': n_classes,
    'n_channels': n_channels,
    'test_accuracy': test_acc,
    'history': history,
}, model_path)

print(f"Model saved to: {model_path}")

## 8. (Optional) Run with Quantum Simulation

**Warning:** Quantum simulation is much slower than classical simulation.

In [ ]:
# Uncomment to run with quantum simulation
# WARNING: This is very slow!

# RUN_QUANTUM = False  # Set to True to enable
# 
# if RUN_QUANTUM:
#     print("Building quantum model...")
#     q_model, q_trainer = build_SEQNN_model(
#         n_classes=n_classes,
#         n_channels=n_channels,
#         use_quantum=True,
#         use_gpu=USE_GPU
#     )
#     
#     # Train on small subset for testing
#     q_trainer.compile(learning_rate=0.01)
#     q_history = q_trainer.fit(
#         train_x[:100], train_y[:100],  # Small subset
#         val_x=valid_x[:50], val_y=valid_y[:50],
#         epochs=5,
#         batch_size=10,
#         verbose=1
#     )

## Summary

This notebook demonstrates the SEQNN model implementation with:

1. **Corrected quantum circuit** matching paper specifications
2. **GPU optimization** for faster training
3. **Proper preprocessing** following paper methodology
4. **Complete training pipeline** with evaluation

For full paper reproduction, ensure:
- Run for 200 epochs (not quick test mode)
- Use the actual datasets (SAT-6, So2Sat LCZ42, Overhead-MNIST)
- Run 3 trials and report mean ± std (as in paper)